# DriveSense-VLM — 02: Model Optimization

**GPU**: A100 80GB (required) | **Time**: ~1.5 h | **Cost**: ~18 CU

Runs the full 3-stage optimization pipeline:
1. **LoRA merge** — fold adapter weights into base model (bfloat16)
2. **AWQ 4-bit quantization** — LLM decoder only; ViT stays fp16
3. **TensorRT ViT compilation** — ONNX → FP16 engine (falls back to `torch.compile`)

Each stage is **idempotent** — skips if its sentinel file already exists.

> ⚠️ **Before running**: Runtime → Change runtime type → **A100 GPU**
>
> **Prerequisites**: `01_training.ipynb` must have saved a checkpoint to Drive.

In [ ]:
# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths ──
PROJECT_ROOT = "/content/drive/MyDrive/DriveSense-VLM"
REPO_ROOT    = "/content/DriveSense-VLM"
DATA_ROOT    = f"{PROJECT_ROOT}/data"
OUTPUTS_ROOT = f"{PROJECT_ROOT}/outputs"

# Create Drive directories
for d in [DATA_ROOT, f"{DATA_ROOT}/nuscenes", f"{DATA_ROOT}/dada2000",
          OUTPUTS_ROOT, f"{OUTPUTS_ROOT}/data", f"{OUTPUTS_ROOT}/training",
          f"{OUTPUTS_ROOT}/data/sft_ready"]:
    os.makedirs(d, exist_ok=True)

# Clone repo to fast ephemeral SSD
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/jayanth922/DriveSense-VLM.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
os.chdir(REPO_ROOT)

# Symlink data/outputs → Drive (persistent across disconnects)
!ln -sfn {DATA_ROOT} {REPO_ROOT}/data
!ln -sfn {OUTPUTS_ROOT} {REPO_ROOT}/outputs

# Add src to Python path (replaces broken editable install)
sys.path.insert(0, f"{REPO_ROOT}/src")
print(f"✓ Repo:  {REPO_ROOT}")
print(f"✓ Drive: {PROJECT_ROOT}")

In [ ]:
# Note: nuscenes-devkit (used in nb00/05) must be installed with --no-deps
#       to skip its strict numpy version pin. Not needed in this notebook.
# Install optimization dependencies
!pip install --upgrade pip setuptools wheel -q 2>&1 | tail -1
!pip install pyyaml tqdm Pillow requests -q 2>&1 | tail -1
!pip install transformers peft accelerate -q 2>&1 | tail -1
!pip install autoawq -q 2>&1 | tail -1
# TensorRT: best-effort (falls back to torch.compile if unavailable)
!pip install tensorrt -q 2>&1 | tail -1 || echo "TensorRT not available — torch.compile fallback will be used"

# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU! Set Runtime → A100 GPU"
print(f"✓ GPU : {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import drivesense
print(f"✓ drivesense importable")

In [ ]:
# Verify GPU and locate training checkpoint
import os, glob, torch

assert torch.cuda.is_available(), "No GPU — set Runtime → A100 GPU"

TRAIN_OUT = f"{OUTPUTS_ROOT}/training"
best = f"{TRAIN_OUT}/checkpoint-best"
if not os.path.exists(best):
    ckpts = sorted(glob.glob(f"{TRAIN_OUT}/checkpoint-*"))
    best = ckpts[-1] if ckpts else None

if best and os.path.exists(best):
    ADAPTER_PATH = best
    print(f"✓ Adapter: {ADAPTER_PATH}")
    !ls {ADAPTER_PATH}
else:
    raise FileNotFoundError(
        f"No checkpoint found in {TRAIN_OUT}\n"
        "Run 01_training.ipynb first."
    )

In [ ]:
# Run full optimization: merge → AWQ quantize → TensorRT ViT
# Each stage is idempotent (skips if output sentinel exists).
# RECOVERY: rerun Cells 2-3 after disconnect, then rerun this cell.
!python scripts/run_optimize_model.py --all \
    --adapter-path {ADAPTER_PATH} \
    --override inference.merge.output_dir={OUTPUTS_ROOT}/merged_model \
    --override inference.quantization.output_dir={OUTPUTS_ROOT}/quantized_model \
    --override inference.tensorrt.output_dir={OUTPUTS_ROOT}/tensorrt \
    2>&1

In [ ]:
# Display optimization results
import os, json

print("=" * 60)
print("  Optimization Results")
print("=" * 60)

merged = f"{OUTPUTS_ROOT}/merged_model"
quant  = f"{OUTPUTS_ROOT}/quantized_model"
trt    = f"{OUTPUTS_ROOT}/tensorrt"

for label, path, sentinel in [
    ("Merged model",     merged, "config.json"),
    ("AWQ quantized",    quant,  "quant_config.json"),
    ("TensorRT / fallback", trt, "optimization_report.txt"),
]:
    sentinel_path = f"{path}/{sentinel}"
    if os.path.exists(sentinel_path):
        size = sum(f.stat().st_size for f in __import__('pathlib').Path(path).rglob('*') if f.is_file()) / 1e9
        print(f"  ✓  {label:<22}: {size:.2f} GB")
        if label == "AWQ quantized" and os.path.exists(f"{path}/quant_config.json"):
            cfg = json.loads(open(f"{path}/quant_config.json").read())
            print(f"       bits={cfg.get('bits','?')}  zero_point={cfg.get('zero_point','?')}")
    else:
        print(f"  ✗  {label:<22}: missing ({sentinel_path})")

trt_engine   = f"{trt}/vit.engine"
trt_fallback = f"{trt}/fallback_info.json"
if os.path.exists(trt_engine):
    print(f"\n  ✓  TensorRT engine: {os.path.getsize(trt_engine)/1e6:.1f} MB")
elif os.path.exists(trt_fallback):
    fb = json.loads(open(trt_fallback).read())
    print(f"\n  ✓  TRT fallback: {fb.get('method','unknown')}")

print("\nProceed to 03_benchmark.ipynb")